<a href="https://colab.research.google.com/github/ggalanc/proyecto_dengue/blob/main/notebooks/practico_06_proyecto_final_fase3_fase5_colab_Gerardo_Galan_Mabel_Herrera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# Práctico Final: MLOps Local con Monitoreo de Data Drift
## Fase 3 y Fase 5 — Servicio de Inferencia, Dashboard y Plan de Acción (Google Colab)
### Magíster en Ciencia de Datos — Tópicos en Data Science II
---

**Integrantes:** Gerardo Galán - Mabel Herrera

**Continúa de:** Fase 2 (`models/modelo_dengue.joblib`, ya entrenado) y Fase 4 (`reports/*.csv`, métricas de drift ya calculadas).

**Qué hace este notebook:**
1. Levanta el servicio de inferencia (FastAPI) de la Fase 3 dentro de esta misma sesión de Colab.
2. Lo alimenta con la producción simulada, exactamente como en una demo local.
3. Prueba en vivo el gatillo de reentrenamiento y el rollback de la Fase 5.
4. Muestra el dashboard de monitoreo de forma inline en el notebook (mismos datos y gráficos que `dashboard/dashboard.py`, sin depender de Streamlit ni de un túnel).


---
### Nota sobre por qué esto corre distinto en Colab que en una máquina local

Localmente, el servicio, el dashboard y el script que los alimenta corren como **procesos separados** en distintas terminales. Colab es un solo notebook con un solo proceso, así que:

- El servicio FastAPI se levanta en un **hilo en segundo plano** dentro de esta misma sesión, en vez de una terminal aparte.
- El dashboard se muestra de forma **inline** en el notebook (los mismos datos que lee `dashboard/dashboard.py`, graficados con matplotlib/pandas) en vez de como app web, porque exponer Streamlit con un túnel público resultó poco confiable entre sesiones de Colab.
- Esta sesión de Colab es **efímera**: todo lo que se genera (modelo reentrenado, base de datos, reportes) vive solo mientras dure esta ejecución. Por eso este notebook está pensado para correrse de principio a fin en una sola sesión continua — igual que será la demo en vivo de la defensa oral — y no para dejar resultados guardados entre sesiones distintas.


---
## Setup — Traer el repositorio a Colab e instalar dependencias


In [ ]:
# ── Bootstrap para Google Colab: trae el repositorio a este entorno ──
import os

REPO_URL = "https://github.com/ggalanc/proyecto_dengue.git"
REPO_DIR = "/content/dengue_mlops_drift_repo_1"

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB and not os.path.exists(REPO_DIR):
    if REPO_URL:
        print("Clonando repositorio...")
        get_ipython().system('git clone -q "{}" "{}"'.format(REPO_URL, REPO_DIR))
    else:
        print("REPO_URL esta vacio. Selecciona el .zip del repositorio para subirlo:")
        from google.colab import files
        subido = files.upload()
        zip_name = list(subido.keys())[0]
        os.makedirs(REPO_DIR, exist_ok=True)
        get_ipython().system('unzip -q "{}" -d "{}"'.format(zip_name, REPO_DIR))
        contenido = os.listdir(REPO_DIR)
        if len(contenido) == 1 and os.path.isdir(os.path.join(REPO_DIR, contenido[0])):
            sub = os.path.join(REPO_DIR, contenido[0])
            for item in os.listdir(sub):
                os.rename(os.path.join(sub, item), os.path.join(REPO_DIR, item))
            os.rmdir(sub)

if IN_COLAB:
    os.chdir(REPO_DIR)  # aqui NO entramos a notebooks/ -- necesitamos service/ y dashboard/ como hermanos
    print("Directorio de trabajo:", os.getcwd())
else:
    print("No se detecto Colab -- se asume ejecucion local normal, sin cambios de directorio.")


In [ ]:
# ── Dependencias que Colab NO trae preinstaladas ──
# (pandas, numpy, scipy, scikit-learn, matplotlib, seaborn y joblib ya vienen listos en Colab)
get_ipython().system('pip install -q fastapi "uvicorn[standard]" httpx pydantic streamlit')
print("Dependencias instaladas")


---
## Fase 3 — Levantar el servicio de inferencia dentro de Colab

Usamos el modelo que ya viene entrenado en `models/modelo_dengue.joblib` (Fase 2) — no hace falta reentrenar para esta demo. `service/app.py` es exactamente el mismo archivo que se usaría con `uvicorn app:app --port 8000` en una máquina local; aquí simplemente lo corremos en un hilo en vez de en una terminal aparte.


In [ ]:
# ── Levantar el servicio FastAPI en un hilo de fondo (version robusta) ──
import sys, threading, time
import requests

sys.path.insert(0, os.path.join(REPO_DIR, "service"))
sys.path.insert(0, REPO_DIR)
os.chdir(os.path.join(REPO_DIR, "service"))

from app import app as fastapi_app
import uvicorn

HEALTH_URL = "http://127.0.0.1:8000/health"

def _iniciar_servicio():
    uvicorn.run(fastapi_app, host="127.0.0.1", port=8000, log_level="warning")

def _servicio_responde():
    try:
        r = requests.get(HEALTH_URL, timeout=2)
        return r
    except requests.exceptions.ConnectionError:
        return None

# si esta celda ya se corrio antes en esta misma sesion, reusamos el servicio en vez de duplicarlo
r = _servicio_responde()
if r is not None:
    print("El servicio ya estaba corriendo:", r.status_code)
    print(r.json())
else:
    server_thread = threading.Thread(target=_iniciar_servicio, daemon=True)
    server_thread.start()

    # Colab a veces tarda en arrancar el servicio (mas si se acaban de instalar
    # dependencias en esta misma sesion) -- reintentamos hasta 20 segundos en vez
    # de asumir un tiempo fijo.
    r = None
    for _ in range(20):
        time.sleep(1)
        r = _servicio_responde()
        if r is not None:
            break

    if r is not None:
        print("Estado del servicio:", r.status_code)
        print(r.json())
    else:
        print(
            "El servicio no respondio despues de 20 segundos.\n"
            "Esto suele pasar la PRIMERA vez que se instalan dependencias nuevas en esta sesion.\n"
            "Solucion: Entorno de ejecucion > Reiniciar sesion, y despues Entorno de ejecucion > Ejecutar todas."
        )


---
## Alimentar el servicio con la producción simulada (demo real de Fase 3)

Corremos `service/simular_produccion.py` sin modificarlo — le manda al servicio, semana por semana, las 292 observaciones de `data/processed/produccion_simulada.csv`, exactamente como en la demo local.


In [ ]:
# ── Alimentar el servicio (puede tardar 1-2 minutos: son 292 peticiones reales) ──
if _servicio_responde() is None:
    print(
        "El servicio no responde en http://127.0.0.1:8000 -- vuelvan a ejecutar la celda "
        "anterior ('Levantar el servicio FastAPI...') antes de continuar."
    )
else:
    os.chdir(os.path.join(REPO_DIR, "service"))
    get_ipython().system('python simular_produccion.py --url http://127.0.0.1:8000')


In [ ]:
# ── Confirmar cuántas predicciones quedaron registradas ──
r = requests.get("http://127.0.0.1:8000/health")
print(r.json())


---
## Puente a la Fase 4 (métricas de drift)

El cálculo de PSI/KS por ventana y las alertas (`reports/drift_detalle.csv`, `reports/mae_por_ventana.csv`, `reports/alertas_por_ventana.csv`) ya está hecho y viene incluido en el repositorio — el dashboard de más abajo simplemente los lee, igual que en la versión local.

Si quieren **regenerarlos desde cero** en esta misma sesión de Colab (por ejemplo, porque acaban de reentrenar el modelo más abajo y quieren que el dashboard refleje el modelo nuevo), corran esta celda — ejecuta el notebook de Fase 4 completo de forma no interactiva, igual que sugiere el README para la opción de reentrenar desde cero:


In [ ]:
# ── OPCIONAL: regenerar reports/ ejecutando el notebook de Fase 4 completo ──
REGENERAR_REPORTS = False  # cambien a True si quieren recalcular el drift con el modelo actual

if REGENERAR_REPORTS:
    os.chdir(REPO_DIR)
    nb = "notebooks/practico_06_proyecto_final_fase4_drift_Gerardo_Galan_Mabel_Herrera.ipynb"
    get_ipython().system('jupyter nbconvert --to notebook --execute --inplace "{}"'.format(nb))
    print("reports/ regenerado con el modelo actual")
else:
    print("Se usan los reports/ ya incluidos en el repositorio (sin regenerar)")


---
## Fase 5 — Probar el gatillo de reentrenamiento y el rollback

Esto es exactamente `service/reentrenar.py`, sin modificar — el mismo script que se documenta en `PLAN_DE_ACCION.md`. Como esta copia del repositorio vive solo dentro de esta sesión de Colab, no hay ningún riesgo de tocar el modelo real del repositorio local: pueden correr esto las veces que quieran.


In [ ]:
# ── Ver el estado actual de alertas por ventana (Fase 4) ──
import pandas as pd
os.chdir(REPO_DIR)
pd.read_csv("reports/alertas_por_ventana.csv")


In [ ]:
# ── Reentrenar (ejemplo: San Juan, ventanas V2 y V3 -- ajustar segun lo que quieran demostrar) ──
os.chdir(os.path.join(REPO_DIR, "service"))
get_ipython().system('python reentrenar.py --city sj --ventanas V2 V3')


In [ ]:
# ── Ver el historial de versiones generado ──
import json
with open(os.path.join(REPO_DIR, "models", "historial_versiones.json")) as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))


In [ ]:
# ── Probar el rollback ──
os.chdir(os.path.join(REPO_DIR, "service"))
get_ipython().system('python reentrenar.py --rollback')


---
## Dashboard de monitoreo — vista inline en el notebook (sin Streamlit, sin túnel)

Después de varias pruebas, exponer Streamlit desde Colab con un túnel público resultó
poco confiable (proxy de Colab que no sostiene el WebSocket, problemas de buffering con
`localtunnel`, y el propio `streamlit` sin quedar bien instalado en el PATH según la sesión).
En vez de seguir dependiendo de eso, las celdas de abajo dibujan **exactamente los mismos
datos que muestra `dashboard/dashboard.py`** (alertas, MAE por ventana, drift por variable,
predicho vs. real) directamente en el notebook, con `pandas`/`matplotlib`/`seaborn` — todo ya
instalado en Colab, sin ningún servicio ni túnel de por medio.

El dashboard interactivo completo (con filtros y navegación) sigue siendo la forma de verlo
en la **demo local** (`streamlit run dashboard.py`, ver el README) — eso ya está probado y
funciona sin problemas en menos de 10 minutos.


In [ ]:
# ── Cargar los mismos datos que lee dashboard/dashboard.py ──
import json, sqlite3
import pandas as pd

sys.path.insert(0, REPO_DIR)
from src.drift import UMBRAL_PSI_SIGNIFICATIVO, UMBRAL_PSI_MODERADO

MODELS_DIR = os.path.join(REPO_DIR, "models")
REPORTS_DIR = os.path.join(REPO_DIR, "reports")
DB_PATH = os.path.join(REPO_DIR, "logs", "dengue_service.db")
PROD_CSV = os.path.join(REPO_DIR, "data", "processed", "produccion_simulada.csv")

CIUDAD = "sj"  # cambiar a "iq" para ver Iquitos y volver a correr las celdas de abajo

with open(os.path.join(MODELS_DIR, "metadata.json")) as f:
    metadata = json.load(f)

drift = pd.read_csv(os.path.join(REPORTS_DIR, "drift_detalle.csv"))
mae = pd.read_csv(os.path.join(REPORTS_DIR, "mae_por_ventana.csv"))
alertas = pd.read_csv(os.path.join(REPORTS_DIR, "alertas_por_ventana.csv"))

preds = pd.DataFrame()
if os.path.exists(DB_PATH):
    conn = sqlite3.connect(DB_PATH)
    preds = pd.read_sql("SELECT * FROM predicciones ORDER BY week_start_date", conn)
    conn.close()
    if len(preds):
        preds["week_start_date"] = pd.to_datetime(preds["week_start_date"])

prod_real = pd.DataFrame()
if os.path.exists(PROD_CSV):
    prod_real = pd.read_csv(PROD_CSV, parse_dates=["week_start_date"])[["city", "week_start_date", "total_cases"]]

print("Monitoreo de Drift -- DengAI (San Juan / Iquitos)")
print("Ciudad seleccionada:", "San Juan" if CIUDAD == "sj" else "Iquitos")
print("Modelo entrenado hasta:", metadata["fecha_entrenamiento_referencia"])
print("MAE de validacion (referencia):", round(metadata["mae_validacion_timeseriessplit"], 2))
print("Predicciones registradas:", len(preds))


In [ ]:
# ── Alertas por ventana ──
from IPython.display import display

alertas_c = alertas[alertas.city == CIUDAD].sort_values("ventana")

def _color_accion(val):
    color = {"OK": "#2ecc71", "REVISAR": "#f39c12", "REENTRENAR": "#e74c3c"}.get(val, "white")
    return f"background-color: {color}; color: white; font-weight: bold"

try:
    estilo = alertas_c.style.map(_color_accion, subset=["accion_sugerida"])  # pandas >= 2.1
except AttributeError:
    estilo = alertas_c.style.applymap(_color_accion, subset=["accion_sugerida"])  # pandas < 2.1

display(estilo)


In [ ]:
# ── Desempeño del modelo (MAE) por ventana ──
import matplotlib.pyplot as plt

mae_c = mae[mae.city == CIUDAD].sort_values("ventana")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(mae_c["ventana"], mae_c["MAE"], marker="o", linewidth=2, color="#e74c3c", label="MAE en produccion")
ax.axhline(metadata["mae_validacion_timeseriessplit"], color="gray", linestyle="--",
           label="MAE de validacion (referencia)")
ax.set_ylabel("MAE")
ax.set_title("Desempeño del modelo por ventana")
ax.legend()
plt.show()


In [ ]:
# ── Drift por variable (PSI) ──
import seaborn as sns

print(
    f"PSI < {UMBRAL_PSI_MODERADO} = sin drift relevante | "
    f"{UMBRAL_PSI_MODERADO}-{UMBRAL_PSI_SIGNIFICATIVO} = drift moderado (monitorear) | "
    f"> {UMBRAL_PSI_SIGNIFICATIVO} = drift significativo"
)

drift_c = drift[drift.city == CIUDAD]
piv = drift_c.pivot(index="feature", columns="ventana", values="psi").reindex(columns=["V1", "V2", "V3", "V4"])

fig2, ax2 = plt.subplots(figsize=(10, 5))
sns.heatmap(piv, annot=True, fmt=".2f", cmap="RdYlGn_r", center=UMBRAL_PSI_SIGNIFICATIVO,
            vmin=0, vmax=max(1.0, float(piv.values.max())), ax=ax2, cbar_kws={"label": "PSI"})
ax2.set_title("Drift por variable (PSI)")
plt.show()


In [ ]:
# ── Predicho vs. real ──
if len(preds):
    p_c = preds[preds.city == CIUDAD].sort_values("week_start_date")
    m = p_c.merge(prod_real[prod_real.city == CIUDAD], on=["city", "week_start_date"], how="left")

    fig3, ax3 = plt.subplots(figsize=(12, 4))
    ax3.plot(m["week_start_date"], m["total_cases"], label="Real", color="#2c3e50", linewidth=1.5)
    ax3.plot(m["week_start_date"], m["total_cases_predicho"], label="Predicho", color="#e74c3c",
              linewidth=1.5, linestyle="--")
    ax3.set_ylabel("Casos por semana")
    ax3.set_title("Predicho vs. real")
    ax3.legend()
    plt.show()

    display(m[["week_start_date", "total_cases", "total_cases_predicho",
               "cobertura_historica_suficiente"]])
else:
    print(
        "No hay predicciones registradas todavia -- corran antes la celda que alimenta "
        "el servicio con la produccion simulada."
    )


---
## Cierre y limitaciones de esta versión en Colab

- Todo lo generado en esta sesión (modelo reentrenado, base de datos de predicciones, historial de versiones) **se pierde al cerrar o reiniciar el entorno de ejecución** de Colab — es intencional (ver la nota de la Fase 5 más arriba); para conservarlo entre sesiones habría que montar Google Drive, que decidimos no hacer para mantener esto simple.
- El dashboard se muestra de forma inline (`matplotlib`/`pandas`, sin `streamlit`) porque exponerlo como app web desde Colab con un túnel público resultó poco confiable entre sesiones; localmente sí se ve como app interactiva completa (`streamlit run dashboard.py`, ver el README).
- Todo el código que genera los datos que se grafican aquí (`service/app.py`, `service/simular_produccion.py`, `service/reentrenar.py`) es exactamente el mismo que corre localmente — no se duplicó ni se reescribió nada.
